# UC CosMx: Fib-Mac Co-occurrence Validation (Fig 2M–N)

**Part 1** (this notebook, original): Nearest-neighbor co-occurrence odds ratios
between OGN+RSPO3+ Fib and SPP1+ Mac in serial-cut 1K validation experiment.

**Part 2** (slide-adjusted): Mixed effects model with slide as fixed effect
and clustered standard errors.

In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/uc_serial_cuts/1000plex"
OUTPUT_DIR = DATA_DIR + "/Processed_merged/colocal"

In [ ]:
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as npfrom matplotlib.colors import ListedColormap
from scipy import sparse
from scipy.sparse import block_diag
from sklearn.neighbors import NearestNeighbors
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multitest import multipletests
import squidpy as sq
import statsmodels.formula.api as smf


In [ ]:
adata=sc.read_h5ad('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/h5ad_obj/qc_norm_log1p.h5ad')

In [ ]:
noimp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/anno/norm_counts_aucell.csv')

In [ ]:
imp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir//anno/imp_log1p_counts_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/uc/cell_type_name_mapper.csv')

In [ ]:
imp_anno=imp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')
noimp_anno=noimp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')

In [ ]:
adata.obs['v7_ywl_b_imp_cell_type_short'] = imp_anno['cell_type_short'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_full'] = imp_anno['cell_type'].tolist()
adata.obs['v7_ywl_b_imp_cell_category_from_type'] = imp_anno['cell_category'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_general'] = imp_anno['cell_type_general'].tolist()

In [ ]:
adata.obs['cell_type_short'] = noimp_anno['cell_type_short'].tolist()
adata.obs['cell_type_full'] = noimp_anno['cell_type'].tolist()
adata.obs['cell_category_from_type'] = noimp_anno['cell_category'].tolist()
adata.obs['cell_type_general'] = noimp_anno['cell_type_general'].tolist()

In [ ]:
adata.obs['patient'].value_counts()

In [ ]:
adata.obs['slide'].value_counts()

In [ ]:
adata.obsm["spatial"] = adata.obs[['CenterX_local_px', 'CenterY_local_px']].to_numpy()

In [ ]:
del adata.var
del adata.uns
del adata.layers

In [ ]:
adata.obs=adata.obs[['cell_id','patient','slide','v7_ywl_b_imp_cell_type_short', 'v7_ywl_b_imp_cell_type_full', 'v7_ywl_b_imp_cell_category_from_type', 'v7_ywl_b_imp_cell_type_general']]

In [ ]:
adata_paired=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/1k/Processed_merged/h5ad_obj/imp_ywl_anno.h5ad')

In [ ]:
adata_paired_inf = adata_paired[adata_paired.obs['condition']=='Inflamed']

In [ ]:
del adata_paired_inf.uns
del adata_paired_inf.varm
del adata_paired_inf.layers
del adata_paired_inf.obsp

In [ ]:
del adata_paired_inf.obsm['X_pca']
del adata_paired_inf.obsm['X_pca_harmony']
del adata_paired_inf.obsm['X_umap']

In [ ]:
del adata_paired_inf.var

In [ ]:
adata_paired_inf.obs['slide']=adata_paired_inf.obs['tissue'].tolist()

In [ ]:
adata_paired_inf.obs=adata_paired_inf.obs[['cell_id','patient','slide','v7_ywl_b_imp_cell_type_short', 'v7_ywl_b_imp_cell_type_full', 'v7_ywl_b_imp_cell_category_from_type', 'v7_ywl_b_imp_cell_type_general']]

In [ ]:
adata_merged = adata.concatenate(adata_paired_inf, 
                                  batch_key="batch",  # adds a column in .obs
                                  batch_categories=["ym", "paired"],
                                  join="inner") 

In [ ]:
adata_sub_mye_fib = adata_merged[adata_merged.obs['v7_ywl_b_imp_cell_category_from_type'].isin(['Myeloid','Connective'])]
adata_sub_mye_fib

In [ ]:
fib_mye_grouped =[]
for i in adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_type_short'].tolist():
    if i in ['Mac S+XS-','Mac S+SG+']:
        fib_mye_grouped.append('Mac S+SG+')
    elif i in ['cDC1','cDC2','CCR7+ DC']:
        fib_mye_grouped.append('DC')
    elif 'Peri' in i:
        fib_mye_grouped.append('Pericyte')
    elif 'Myofib' in i:
        fib_mye_grouped.append('Myofib')
    elif 'Crypt Top' in i:
        fib_mye_grouped.append('Crypt Top Fib')
    else:
        fib_mye_grouped.append(i)

In [ ]:
adata_sub_mye_fib.obs['fib_mye_grouped']=fib_mye_grouped

In [ ]:
adata_sub_mye_fib = adata_sub_mye_fib[~adata_sub_mye_fib.obs['fib_mye_grouped'].isin(['Cycl Myeloid','Cycl Fib','Pericyte','SELENOP+ Fib'])]

In [ ]:

adata_sub_mye_fib.obs['fib_mye_grouped'] = adata_sub_mye_fib.obs['fib_mye_grouped'].astype('category')

In [ ]:
adata_sub_mye_fib.obs["slide"].unique()

In [ ]:
adata_sub_fib=adata_sub_mye_fib[adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_category_from_type']=='Connective']

In [ ]:
adata_sub_fib.obs['v7_ywl_b_imp_cell_type_short'].unique()

In [ ]:
adata_sub_mac=adata_sub_mye_fib[adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_category_from_type']=='Myeloid']

In [ ]:
adata_sub_mac.obs['v7_ywl_b_imp_cell_type_short'].value_counts()

In [ ]:
act_fib_lst=[]
mac_dens_lst = []
sample_lst = []


for i in adata.obs['slide'].unique():
    print(i)
    adata_sub = adata[adata.obs['slide']==i]
    coords = adata_sub.obsm['spatial']
    nbrs = NearestNeighbors(n_neighbors=20).fit(coords)
    distances, indices = nbrs.kneighbors(coords)
    labels = adata_sub.obs["v7_ywl_b_imp_cell_type_short"].values
    is_mac = np.isin(labels, ['Mo-Mac','Inf Mo-Mac','Mac S+M+S+'])
    mac_density = [sum(is_mac[neighbor_ids]) for neighbor_ids in indices]
    is_fib = np.isin(labels, adata_sub_fib.obs['v7_ywl_b_imp_cell_type_short'].unique().tolist())
    fib_coords = coords[is_fib]
    fib_density = np.array(mac_density)[is_fib]
    act_fib = np.isin(labels, ['Activ Fib']).astype(int)
    act_fib_density = np.array(act_fib)[is_fib]
    sample = [i] * len(act_fib_density)
    act_fib_lst.append(act_fib_density)
    mac_dens_lst.append(fib_density)
    sample_lst.append(sample)

In [ ]:
df = pd.DataFrame({
    "act_fib": [item for sublist in act_fib_lst for item in sublist],
    "inc_mac_density": [item for sublist in mac_dens_lst for item in sublist],
    "sample": [item for sublist in sample_lst for item in sublist],
})

In [ ]:
model = mixedlm("act_fib ~ inc_mac_density", df, groups=df["sample"])
result = model.fit()
print(result.summary())

In [ ]:
# Grouped categories
selenop_pos_mac_types = ['Mac S+XS-', 'Mac S+SG+']
dc_types = ['cDC1', 'cDC2', 'CCR7+ DC']
#momac_types = []

# Individual cell types you want to keep separate
indiv_cell_types = ['Mo-Mac','Inf Mo-Mac', 
    'Mac S+M+S+', 'Neutrophil', 'Mac S+M+P+',
    'Trans cDC2/Mac']

# Initialize all output lists
act_fib_lst = []
sample_lst = []
mac_dens_selenop_lst = []
mac_dens_dc_lst = []
#mac_dens_mo_lst = []
indiv_cell_densities = {ctype: [] for ctype in indiv_cell_types}


In [ ]:
for i in adata.obs['slide'].unique():
    print(i)
    adata_sub = adata[adata.obs['slide'] == i]
    coords = adata_sub.obsm['spatial']
    nbrs = NearestNeighbors(n_neighbors=20).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    labels = adata_sub.obs["v7_ywl_b_imp_cell_type_short"].values

    # Create masks
    is_selenop = np.isin(labels, selenop_pos_mac_types)
    is_dc = np.isin(labels, dc_types)
    #is_mo = np.isin(labels, momac_types)
    
    # Precompute group densities
    selenop_density = [sum(is_selenop[neighbor_ids]) for neighbor_ids in indices]
    dc_density = [sum(is_dc[neighbor_ids]) for neighbor_ids in indices]
    #mo_density = [sum(is_mo[neighbor_ids]) for neighbor_ids in indices]

    # Compute individual cell type densities
    indiv_density_by_type = {}
    for ctype in indiv_cell_types:
        is_target = (labels == ctype)
        indiv_density_by_type[ctype] = [sum(is_target[neighbor_ids]) for neighbor_ids in indices]

    # Restrict to fibroblasts
    is_fib = np.isin(labels, adata_sub_fib.obs['v7_ywl_b_imp_cell_type_short'].unique().tolist())

    # Subset data for fibroblasts only
    act_fib = np.isin(labels, ['Activ Fib']).astype(int)
    act_fib_density = np.array(act_fib)[is_fib]
    sample = [i] * len(act_fib_density)

    # Store outputs
    act_fib_lst.append(act_fib_density)
    mac_dens_selenop_lst.append(np.array(selenop_density)[is_fib])
    mac_dens_dc_lst.append(np.array(dc_density)[is_fib])
    #mac_dens_mo_lst.append(np.array(mo_density)[is_fib])
    sample_lst.append(sample)

    # Append densities for each cell type
    for ctype in indiv_cell_types:
        indiv_cell_densities[ctype].append(np.array(indiv_density_by_type[ctype])[is_fib])


In [ ]:
# Start with main columns
data = {
    "act_fib": np.concatenate(act_fib_lst),
    "density_Mac_SEL": np.concatenate(mac_dens_selenop_lst),
    "density_DC": np.concatenate(mac_dens_dc_lst),
    #"density_Mo_Mac": np.concatenate(mac_dens_mo_lst),
    "sample": np.concatenate(sample_lst),
}

# Add each individual cell type density column
for ctype in indiv_cell_types:
    colname = f"density_{ctype.replace(' ', '_').replace('+', '').replace('/','_')}"
    data[colname] = np.concatenate(indiv_cell_densities[ctype])

df = pd.DataFrame(data)


In [ ]:

# Rename columns before modeling
df_model = df.copy()
df_model.columns = df_model.columns.str.replace('-', '_')
df_model

In [ ]:


formula = "act_fib ~ density_Inf_Mo_Mac + density_Mo_Mac + density_Neutrophil + density_DC + " \
          "density_Mac_SMS + density_Mac_SMP + density_Mac_SEL + density_Trans_cDC2_Mac"


# Option A: WITHOUT sample fixed effects (just cluster SEs)

res = smf.logit(formula, data=df_model).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": df["sample"]}
)
'''
# Option B: WITH sample fixed effects (and cluster SEs)
res = smf.logit(formula + " + C(sample)", data=df_model).fit(
     disp=False,
     cov_type="cluster",
     cov_kwds={"groups": df_model["sample"]}
)
'''
# Odds ratios and 95% CI (uses clustered covariance)
or_ = np.exp(res.params)
ci = np.exp(res.conf_int())
p  = res.pvalues

or_table = pd.DataFrame({"OR": or_, "CI_low": ci[0], "CI_high": ci[1], "Pval": p})
or_table


In [ ]:

pvals = [0.000004,0.004140, 0.071069, 0.030910, 0.827986,
         0.013296, 0.124564, 0.490599, 0.021649]

_, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')
print(pvals_fdr)

In [ ]:
or_table=pd.read_csv('/data/ydn4687/ibd/uc_yiming_1k_for_imp/or_table_fib_activation.csv')

In [ ]:
or_table.index = or_table['Unnamed: 0'].tolist()

In [ ]:
or_table['adj_p']=pvals_fdr

In [ ]:
or_table.to_csv('or_table_fib_activation.csv')

In [ ]:
colormap = ListedColormap(custom_palette[1:9])

In [ ]:

# Your existing code...
cell = or_table.loc[or_table.index.str.startswith("density_")].copy()

label_map = {
    "density_Mo_Mac": "Mo-Mac",
    "density_Inf_Mo_Mac": "Inf Mo-Mac",
    "density_Neutrophil": "Neutrophil",
    "density_DC": "DC",
    "density_Mac_SMS": "Mac S+M+S+",
    "density_Mac_SMP": "Mac S+M+P+",
    "density_Mac_SEL": "Mac S+SG+",
    "density_Trans_cDC2_Mac": "Trans cDC2/Mac"
}
cell["Predictor"] = [label_map.get(i, i.replace("density_", "")) for i in cell.index]

# Sort for display
cell = cell.sort_values("OR", ascending=False)

# Get alphabetically sorted unique predictors and assign colors from colormap
sorted_predictors = sorted(cell["Predictor"].unique())
color_dict = dict(zip(sorted_predictors, colormap.colors[:len(sorted_predictors)]))

# Map colors to each row
colors = [color_dict[pred] for pred in cell["Predictor"]]

# Build asymmetric x-errors from CI
xerr = np.vstack([cell["OR"] - cell["CI_low"], cell["CI_high"] - cell["OR"]])

# Significance threshold
sig_thresh = 0.1

# Plot
fig, ax = plt.subplots(figsize=(4, 5))

# Error bars
ax.errorbar(
    cell["OR"],
    cell["Predictor"],
    xerr=xerr,
    fmt='o',
    color='black',
    ecolor='gray',
    capsize=3,
    linestyle="none",
    markerfacecolor="none",
    markeredgecolor="none"
)

# Points with colors from colormap
ax.scatter(
    cell["OR"],
    cell["Predictor"],
    c=colors,
    s=70,
    edgecolor="black"
)

# Add stars for significant results
for i, (idx, row) in enumerate(cell.iterrows()):
    if row["adj_p"] < sig_thresh:
        ax.text(
            row["CI_high"] + 0.03,
            i,
            '*',
            va='center',
            ha='left',
            fontsize=16,
            fontweight='bold'
        )

# Reference line at OR = 1
ax.axvline(1, color='red', linestyle='--')

# Labels
plt.xlim(0.8,1.9)
ax.set_xlabel("OR for Fib Activation", fontsize=13)
ax.set_ylabel("Cell Type (Neighborhood Density)", fontsize=14)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## Part 2: Slide-adjusted Fib-Mac Co-occurrence (Fixed Effects + Clustered SEs)

Mixed effects model with slide as fixed effect and clustered standard errors.
Source: `valid_1k_imp_anno_fib_mac_colocal_slide_adj.ipynb`

In [ ]:
adata=sc.read_h5ad('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/h5ad_obj/qc_norm_log1p.h5ad')


In [ ]:
noimp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/anno/norm_counts_aucell.csv')

In [ ]:
imp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir//anno/imp_log1p_counts_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/uc/cell_type_name_mapper.csv')

In [ ]:
imp_anno=imp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')
noimp_anno=noimp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')

In [ ]:
adata.obs['v7_ywl_b_imp_cell_type_short'] = imp_anno['cell_type_short'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_full'] = imp_anno['cell_type'].tolist()
adata.obs['v7_ywl_b_imp_cell_category_from_type'] = imp_anno['cell_category'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_general'] = imp_anno['cell_type_general'].tolist()

In [ ]:
adata.obs['cell_type_short'] = noimp_anno['cell_type_short'].tolist()
adata.obs['cell_type_full'] = noimp_anno['cell_type'].tolist()
adata.obs['cell_category_from_type'] = noimp_anno['cell_category'].tolist()
adata.obs['cell_type_general'] = noimp_anno['cell_type_general'].tolist()

In [ ]:
adata.obs['slide'].value_counts()

In [ ]:
adata.obsm["spatial"] = adata.obs[['CenterX_local_px', 'CenterY_local_px']].to_numpy()

In [ ]:
adata_sub_mye_fib = adata[adata.obs['v7_ywl_b_imp_cell_category_from_type'].isin(['Myeloid','Connective'])]
adata_sub_mye_fib

In [ ]:
fib_mye_grouped =[]
for i in adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_type_short'].tolist():
    if i in ['Mac S+XS-','Mac S+SG+']:
        fib_mye_grouped.append('Mac S+SG+')
    elif i in ['cDC1','cDC2','CCR7+ DC']:
        fib_mye_grouped.append('DC')
    elif 'Peri' in i:
        fib_mye_grouped.append('Pericyte')
    elif 'Myofib' in i:
        fib_mye_grouped.append('Myofib')
    elif 'Crypt Top' in i:
        fib_mye_grouped.append('Crypt Top Fib')
    else:
        fib_mye_grouped.append(i)

In [ ]:
adata_sub_mye_fib.obs['fib_mye_grouped']=fib_mye_grouped

In [ ]:
adata_sub_mye_fib = adata_sub_mye_fib[~adata_sub_mye_fib.obs['fib_mye_grouped'].isin(['Cycl Myeloid','Cycl Fib','Pericyte','SELENOP+ Fib'])]

In [ ]:

adata_sub_mye_fib.obs['fib_mye_grouped'] = adata_sub_mye_fib.obs['fib_mye_grouped'].astype('category')

In [ ]:

# Start with an empty adjacency for the full object
N = adata_sub_mye_fib.n_obs
C = sparse.csr_matrix((N, N))
D = sparse.csr_matrix((N, N))

for piece in adata_sub_mye_fib.obs["slide"].unique():
    idx = np.where(adata_sub_mye_fib.obs["slide"].values == piece)[0]
    ad_sub = adata_sub_mye_fib[idx].copy()
    sq.gr.spatial_neighbors(ad_sub)  # builds piece-internal graph
    # insert the subgraphs on the block corresponding to idx
    C[idx[:, None], idx] = ad_sub.obsp["spatial_connectivities"]
    D[idx[:, None], idx] = ad_sub.obsp["spatial_distances"]

adata_sub_mye_fib.obsp["spatial_connectivities"] = C
adata_sub_mye_fib.obsp["spatial_distances"] = D


In [ ]:
custom_palette = [
    "#FF9999", "#66B3FF", "#99FF99",  "#FFD700", 
    "#FF69B4", "#8A2BE2", "#00BFFF", "#ADFF2F",
    "#FF4500", "#DDA0DD", "#20B2AA", "#778899", 
   "#FFA500","#7B68EE", "#D2691E", "#00FA9A", 
    "#B22222", "#32CD32", 
]
colormap = ListedColormap(custom_palette)
colormap

In [ ]:
adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_type_short'].value_counts()

In [ ]:
adata_sub_mye__act_fib = adata_sub_mye_fib[~adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_type_short'].isin(['GREM2+ Myofib','ADAMDEC1+ Fib',
                                                                                                       'OGN+RSPO3+ Fib','CXCL5+ Activ Fib',
                                                                                                       'HHIP+ Myofib','T-interact Fib',
                                                                                                       'VSTM2A+ Crypt Top Fib','NRG1+ Crypt Top Fib'])]

In [ ]:

# 1) Build per-slide graphs
Cs, Ds, order = [], [], []
for slide, idx in adata_sub_mye__act_fib.obs.groupby("slide").indices.items():
    ad = adata_sub_mye__act_fib[idx].copy()

    # (optional) rebase; translation doesn't change distances
    xy = ad.obsm["spatial_fov"].astype(float)
    ad.obsm["spatial_fov"] = xy - xy.min(axis=0, keepdims=True)

    # neighbors within slide (choose ONE of kNN or radius)
    sq.gr.spatial_neighbors(ad, spatial_key="spatial_fov", coord_type="generic", n_neighs=6)
    # e.g., radius alt: sq.gr.spatial_neighbors(ad, spatial_key="spatial_fov", coord_type="generic", radius=50.0)

    Cs.append(ad.obsp["spatial_connectivities"])
    Ds.append(ad.obsp["spatial_distances"])
    order.extend(idx.tolist())

# 2) Stitch block-diagonal graph on a reordered copy
adata_bd = adata_sub_mye__act_fib[order].copy()
adata_bd.obsp["spatial_connectivities"] = block_diag(Cs, format="csr")
adata_bd.obsp["spatial_distances"]      = block_diag(Ds, format="csr")

# Quick sanity check: distances should now reflect within-slide mins
d = adata_bd.obsp["spatial_distances"].data
print("Min/95th percentile distances (block-diag):", float(d.min()), float(np.percentile(d, 95)))



In [ ]:
# 3) Run co-occurrence (no n_bins/max_dist here)
sq.gr.co_occurrence(adata_bd, cluster_key="fib_mye_grouped")



In [ ]:
# Increase global font size
plt.rcParams.update({
    'font.size': 14,           # base font size for everything
    'axes.titlesize': 14,      # title
    'axes.labelsize': 14,      # axis labels
    'xtick.labelsize': 12,     # x ticks
    'ytick.labelsize': 12,     # y ticks
    'legend.fontsize': 11,     # legend text
    'legend.title_fontsize': 12
})
# Get unique categories in current order
unique_categories = adata_bd.obs["fib_mye_grouped"].cat.categories.tolist()

# Create a colormap with only the colors you need (in the right order)
n_categories = len(unique_categories)
colormap = ListedColormap(custom_palette[:n_categories])

# 4) Plot
sq.pl.co_occurrence(
    adata_bd,
    cluster_key="fib_mye_grouped",
    clusters="Activ Fib",
    palette=colormap
)

fig = plt.gcf()
fig.set_size_inches(4, 5)
ax = plt.gca()
ax.set_title('')
ax.set_ylabel('Co-occurrence Score w/ Activ Fib')
ax.set_xlabel('Distance')
ax.set_xlim(70, 700)

# Legend inside plot
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles, labels,
    title='Cell Type',
    frameon=False,
    loc='upper right',
    bbox_to_anchor=(0.99, 0.99),
        fontsize=9,          # smaller legend text
    title_fontsize=9,    # smaller legend title
    handlelength=1.2,    # optional: shrink legend handles
    handletextpad=0.4    
)

plt.tight_layout()
plt.show()

In [ ]:
adata_sub_fib=adata_sub_mye_fib[adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_category_from_type']=='Connective']

In [ ]:
adata_sub_fib.obs['v7_ywl_b_imp_cell_type_short'].unique()

In [ ]:
adata_sub_mac=adata_sub_mye_fib[adata_sub_mye_fib.obs['v7_ywl_b_imp_cell_category_from_type']=='Myeloid']

In [ ]:
adata_sub_mac.obs['v7_ywl_b_imp_cell_type_short'].value_counts()

In [ ]:
act_fib_lst=[]
mac_dens_lst = []
sample_lst = []


for i in adata.obs['slide'].unique():
    print(i)
    adata_sub = adata[adata.obs['slide']==i]
    coords = adata_sub.obsm['spatial']
    nbrs = NearestNeighbors(n_neighbors=20).fit(coords)
    distances, indices = nbrs.kneighbors(coords)
    labels = adata_sub.obs["v7_ywl_b_imp_cell_type_short"].values
    is_mac = np.isin(labels, ['Mo-Mac','Inf Mo-Mac','Mac S+M+S+'])
    mac_density = [sum(is_mac[neighbor_ids]) for neighbor_ids in indices]
    is_fib = np.isin(labels, adata_sub_fib.obs['v7_ywl_b_imp_cell_type_short'].unique().tolist())
    fib_coords = coords[is_fib]
    fib_density = np.array(mac_density)[is_fib]
    act_fib = np.isin(labels, ['Activ Fib']).astype(int)
    act_fib_density = np.array(act_fib)[is_fib]
    sample = [i] * len(act_fib_density)
    act_fib_lst.append(act_fib_density)
    mac_dens_lst.append(fib_density)
    sample_lst.append(sample)
    

In [ ]:
df = pd.DataFrame({
    "act_fib": [item for sublist in act_fib_lst for item in sublist],
    "inc_mac_density": [item for sublist in mac_dens_lst for item in sublist],
    "sample": [item for sublist in sample_lst for item in sublist],
})

In [ ]:
model = mixedlm("act_fib ~ inc_mac_density", df, groups=df["sample"])
result = model.fit()
print(result.summary())

In [ ]:
# Grouped categories
selenop_pos_mac_types = ['Mac S+XS-', 'Mac S+SG+']
dc_types = ['cDC1', 'cDC2', 'CCR7+ DC']

# Individual cell types you want to keep separate
indiv_cell_types = [
    'Mac S+M+S+', 'Neutrophil', 'Mac S+M+P+',
    'Trans cDC2/Mac','Inf Mo-Mac', 'Mo-Mac']

# Initialize all output lists
act_fib_lst = []
sample_lst = []
mac_dens_selenop_lst = []
mac_dens_dc_lst = []
#mac_dens_mo_lst = []
indiv_cell_densities = {ctype: [] for ctype in indiv_cell_types}


In [ ]:
for i in adata.obs['slide'].unique():
    print(i)
    adata_sub = adata[adata.obs['slide'] == i]
    coords = adata_sub.obsm['spatial']
    nbrs = NearestNeighbors(n_neighbors=20).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    labels = adata_sub.obs["v7_ywl_b_imp_cell_type_short"].values

    # Create masks
    is_selenop = np.isin(labels, selenop_pos_mac_types)
    is_dc = np.isin(labels, dc_types)
    #is_mo = np.isin(labels, momac_types)
    
    # Precompute group densities
    selenop_density = [sum(is_selenop[neighbor_ids]) for neighbor_ids in indices]
    dc_density = [sum(is_dc[neighbor_ids]) for neighbor_ids in indices]
    #mo_density = [sum(is_mo[neighbor_ids]) for neighbor_ids in indices]

    # Compute individual cell type densities
    indiv_density_by_type = {}
    for ctype in indiv_cell_types:
        is_target = (labels == ctype)
        indiv_density_by_type[ctype] = [sum(is_target[neighbor_ids]) for neighbor_ids in indices]

    # Restrict to fibroblasts
    is_fib = np.isin(labels, adata_sub_fib.obs['v7_ywl_b_imp_cell_type_short'].unique().tolist())

    # Subset data for fibroblasts only
    act_fib = np.isin(labels, ['Activ Fib']).astype(int)
    act_fib_density = np.array(act_fib)[is_fib]
    sample = [i] * len(act_fib_density)

    # Store outputs
    act_fib_lst.append(act_fib_density)
    mac_dens_selenop_lst.append(np.array(selenop_density)[is_fib])
    mac_dens_dc_lst.append(np.array(dc_density)[is_fib])
    #mac_dens_mo_lst.append(np.array(mo_density)[is_fib])
    sample_lst.append(sample)

    # Append densities for each cell type
    for ctype in indiv_cell_types:
        indiv_cell_densities[ctype].append(np.array(indiv_density_by_type[ctype])[is_fib])


In [ ]:
# Start with main columns
data = {
    "act_fib": np.concatenate(act_fib_lst),
    "density_Mac_SEL": np.concatenate(mac_dens_selenop_lst),
    "density_DC": np.concatenate(mac_dens_dc_lst),
    #"density_Mo_Mac": np.concatenate(mac_dens_mo_lst),
    "sample": np.concatenate(sample_lst),
}

# Add each individual cell type density column
for ctype in indiv_cell_types:
    colname = f"density_{ctype.replace(' ', '_').replace('+', '').replace('/','_')}"
    data[colname] = np.concatenate(indiv_cell_densities[ctype])

df = pd.DataFrame(data)


In [ ]:

# Rename columns before modeling
df_model = df.copy()
df_model.columns = df_model.columns.str.replace('-', '_')

formula = "act_fib ~ density_Inf_Mo_Mac + density_Mo_Mac + density_Neutrophil + density_DC + " \
          "density_Mac_SMS + density_Mac_SMP + density_Mac_SEL + density_Trans_cDC2_Mac"


# Option A: WITHOUT sample fixed effects (just cluster SEs)
'''
res = smf.logit(formula, data=df).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": df["sample"]}
)
'''
# Option B: WITH sample fixed effects (and cluster SEs)
res = smf.logit(formula + " + C(sample)", data=df_model).fit(
     disp=False,
     cov_type="cluster",
     cov_kwds={"groups": df_model["sample"]}
)
# Odds ratios and 95% CI (uses clustered covariance)
or_ = np.exp(res.params)
ci = np.exp(res.conf_int())
p  = res.pvalues

or_table = pd.DataFrame({"OR": or_, "CI_low": ci[0], "CI_high": ci[1], "Pval": p})
or_table


Fixed effects adjust for mean differences between slides.

Clustered SEs adjust for correlation of errors within slides.

Fixed effects adjust for mean differences between slides.

Clustered SEs adjust for correlation of errors within slides.

In [ ]:
colormap = ListedColormap(custom_palette[1:9])

In [ ]:

# Your existing code...
cell = or_table.loc[or_table.index.str.startswith("density_")].copy()

label_map = {
    "density_Mo_Mac": "Mo-Mac",
    "density_Inf_Mo_Mac": "Inf Mo-Mac",
    "density_Neutrophil": "Neutrophil",
    "density_DC": "DC",
    "density_Mac_SMS": "Mac S+M+S+",
    "density_Mac_SMP": "Mac S+M+P+",
    "density_Mac_SEL": "Mac S+SG+",
    "density_Trans_cDC2_Mac": "Trans cDC2/Mac"
}
cell["Predictor"] = [label_map.get(i, i.replace("density_", "")) for i in cell.index]

# Sort for display
cell = cell.sort_values("OR", ascending=False)

# Get alphabetically sorted unique predictors and assign colors from colormap
sorted_predictors = sorted(cell["Predictor"].unique())
color_dict = dict(zip(sorted_predictors, colormap.colors[:len(sorted_predictors)]))

# Map colors to each row
colors = [color_dict[pred] for pred in cell["Predictor"]]

# Build asymmetric x-errors from CI
xerr = np.vstack([cell["OR"] - cell["CI_low"], cell["CI_high"] - cell["OR"]])

# Significance threshold
sig_thresh = 0.1

# Plot
fig, ax = plt.subplots(figsize=(3.8, 5))

# Error bars
ax.errorbar(
    cell["OR"],
    cell["Predictor"],
    xerr=xerr,
    fmt='o',
    color='black',
    ecolor='gray',
    capsize=3,
    linestyle="none",
    markerfacecolor="none",
    markeredgecolor="none"
)

# Points with colors from colormap
ax.scatter(
    cell["OR"],
    cell["Predictor"],
    c=colors,
    s=70,
    edgecolor="black"
)

# Add stars for significant results
for i, (idx, row) in enumerate(cell.iterrows()):
    if row["Pval"] < sig_thresh:
        ax.text(
            row["CI_high"] + 0.05,
            i,
            '*',
            va='center',
            ha='left',
            fontsize=16,
            fontweight='bold'
        )

# Reference line at OR = 1
ax.axvline(1, color='red', linestyle='--')

# Labels
ax.set_xlabel("OR for Fib Activation", fontsize=13)
ax.set_ylabel("Cell Type (Neighborhood Density)", fontsize=14)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()